# FakeAVCeleb v1.2

Identity-disjoint split, separate visual and audio labels, backbone fine-tuning, audio features and fusion (Section 4.5, Tables 11 and 12).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import pandas as pd
from pathlib import Path

BASE = Path('/content/drive/MyDrive/deepfake_finetuning')
FAV = BASE/'FakeAVCeleb_v1.2'

meta = next((p for p in FAV.rglob('meta_data.csv')), None)
print('meta:', meta)
m = pd.read_csv(meta)
print('\nrows', len(m))
print('columns:', list(m.columns))
print('\n', m.head(5).to_string())
for c in m.columns:
    if m[c].nunique() <= 12:
        print(f'\n{c}:', m[c].value_counts().to_dict())
    else:
        print(f'\n{c}: {m[c].nunique()} unique, e.g. {m[c].dropna().unique()[:4].tolist()}')

# where do the actual video files live?
vids = [p for p in FAV.rglob('*.mp4')]
print(f'\n{len(vids)} mp4 files on disk')
from collections import Counter
for d, n in Counter(str(p.parent) for p in vids).most_common(15):
    print(f'{n:6d}  {d}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
meta: /content/drive/MyDrive/deepfake_finetuning/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/meta_data.csv

rows 21566
columns: ['source', 'target1', 'target2', 'method', 'category', 'type', 'race', 'gender', 'path', 'Unnamed: 9']

     source target1 target2 method category                 type     race gender       path                                           Unnamed: 9
0  id00076       -       -   real        A  RealVideo-RealAudio  African    men  00109.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id00076
1  id00166       -       -   real        A  RealVideo-RealAudio  African    men  00010.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id00166
2  id00173       -       -   real        A  RealVideo-RealAudio  African    men  00118.mp4  FakeAVCeleb/RealVideo-RealAudio/African/men/id00173
3  id00366       -       -   real        A  RealVideo-RealAudio  African

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

BASE=Path('/content/drive/MyDrive/deepfake_finetuning'); OUT=BASE/'rebuild_2026'
FAV=BASE/'FakeAVCeleb_v1.2'/'FakeAVCeleb_v1.2'
SEED=42; rng=np.random.default_rng(SEED)

m=pd.read_csv(FAV/'meta_data.csv')
m['video_path']=m['Unnamed: 9'].str.replace('FakeAVCeleb/','',regex=False)+'/'+m['path']
m['video_path']=m.video_path.apply(lambda r: str(FAV/r))
m['exists']=m.video_path.apply(lambda p: Path(p).exists())
print('files found:', m.exists.sum(), 'of', len(m))
m=m[m.exists].copy()

m['label_visual']=m.type.str.startswith('FakeVideo').astype(int)
m['label_audio']=m.type.str.endswith('FakeAudio').astype(int)
m['group_id']=m.source
m['video_id']=m.type+'__'+m.source+'__'+m.path.str.replace('.mp4','',regex=False)
print('\nlabels by category:')
print(m.groupby(['type','label_visual','label_audio']).size())

# balanced 1,000-clip sample: 500 real + 500 fakes stratified over the three fake types
real=m[m.type=='RealVideo-RealAudio']
fakes=[]
for t,n in [('FakeVideo-FakeAudio',200),('FakeVideo-RealAudio',200),('RealVideo-FakeAudio',100)]:
    sub=m[m.type==t]
    fakes.append(sub.iloc[rng.choice(len(sub),size=min(n,len(sub)),replace=False)])
sample=pd.concat([real]+fakes,ignore_index=True)
print(f'\nsample: {len(sample)} clips, {sample.group_id.nunique()} identities')
print(sample.type.value_counts().to_dict())

# 70/15/15 by IDENTITY
g1=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED)
rest,test=next(g1.split(sample,groups=sample.group_id))
g2=GroupShuffleSplit(n_splits=1,test_size=0.15/0.85,random_state=SEED)
tr,va=next(g2.split(sample.iloc[rest],groups=sample.iloc[rest].group_id))
sample['partition']='train'
sample.iloc[rest[va],sample.columns.get_loc('partition')]='val'
sample.iloc[test,sample.columns.get_loc('partition')]='test'

for a in ('train','val','test'):
    for b in ('train','val','test'):
        if a<b:
            assert not (set(sample[sample.partition==a].group_id) &
                        set(sample[sample.partition==b].group_id)), f'{a}/{b} identity overlap'
print('\nno identity appears in two partitions')
print(sample.groupby('partition').agg(clips=('partition','size'),identities=('group_id','nunique')))
print('\nvisual labels:'); print(sample.pivot_table(index='partition',columns='label_visual',
                                  values='video_id',aggfunc='count',fill_value=0))
print('audio labels:'); print(sample.pivot_table(index='partition',columns='label_audio',
                              values='video_id',aggfunc='count',fill_value=0))
cols=['video_path','video_id','group_id','type','method','label_visual','label_audio','partition']
sample[cols].to_csv(OUT/'manifests'/'fakeavceleb_manifest.csv',index=False)
print('\nwrote fakeavceleb_manifest.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
files found: 21566 of 21566

labels by category:
type                 label_visual  label_audio
FakeVideo-FakeAudio  1             1              10857
FakeVideo-RealAudio  1             0               9709
RealVideo-FakeAudio  0             1                500
RealVideo-RealAudio  0             0                500
dtype: int64

sample: 1000 clips, 500 identities
{'RealVideo-RealAudio': 500, 'FakeVideo-FakeAudio': 200, 'FakeVideo-RealAudio': 200, 'RealVideo-FakeAudio': 100}

no identity appears in two partitions
           clips  identities
partition                   
test         143          75
train        704         350
val          153          75

visual labels:
label_visual    0    1
partition             
test           86   57
train         418  286
val            96   57
audio labels:
label_audio    0    1
partition            
test         104

Face crops and a first audio feature set. The faces from this cell are used. The audio features printed here are superseded by the 40 MFCC, 7 spectral-contrast and 1 ZCR features computed further down (Section 3.3.2).

In [ ]:
!pip -q install --no-deps facenet-pytorch==2.6.0
!pip -q install librosa
from google.colab import drive; drive.mount('/content/drive')
import cv2, numpy as np, pandas as pd, torch, time, librosa, subprocess, tempfile
from pathlib import Path
from PIL import Image
from facenet_pytorch import MTCNN

BASE=Path('/content/drive/MyDrive/deepfake_finetuning'); OUT=BASE/'rebuild_2026'
FR=OUT/'frames'/'fakeav'; FR.mkdir(parents=True,exist_ok=True)
AU=OUT/'features'; AU.mkdir(exist_ok=True)
K,SIZE,N_MFCC=20,300,24          # 24 mean + 24 std = the 48-d representation
man=pd.read_csv(OUT/'manifests'/'fakeavceleb_manifest.csv')
det=MTCNN(image_size=SIZE,margin=20,keep_all=False,post_process=False,
          select_largest=True,device='cuda')
print(len(man),'clips')

def faces(video,out_dir,uid):
    out_dir.mkdir(parents=True,exist_ok=True)
    cap=cv2.VideoCapture(str(video)); n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    want=set(np.linspace(0,max(n-1,0),min(n,K)).round().astype(int).tolist()) if n else set()
    w,i=[],0
    while want and i<=max(want):
        ok,fr=cap.read()
        if not ok: break
        if i in want:
            img=Image.fromarray(cv2.cvtColor(fr,cv2.COLOR_BGR2RGB))
            try: f=det(img)
            except Exception: f=None
            if f is not None:
                a=f.permute(1,2,0).numpy(); a=((a-a.min())/(np.ptp(a)+1e-8)*255).astype('uint8')
                d=out_dir/f'{uid}__f{i:05d}.jpg'
                Image.fromarray(a).resize((SIZE,SIZE)).save(d,quality=95); w.append(str(d))
        i+=1
    cap.release(); return w

def mfcc(video):
    with tempfile.NamedTemporaryFile(suffix='.wav') as t:
        subprocess.run(['ffmpeg','-y','-i',str(video),'-ac','1','-ar','16000',t.name],
                       capture_output=True)
        y,sr=librosa.load(t.name,sr=16000)
    if len(y)<sr*0.1: return np.zeros(N_MFCC*2,dtype='float32')
    c=librosa.feature.mfcc(y=y,sr=sr,n_mfcc=N_MFCC)
    return np.concatenate([c.mean(1),c.std(1)]).astype('float32')

rows,aud,failed,t0=[],[],[],time.time()
for n,r in enumerate(man.itertuples(),1):
    d=FR/r.partition
    done=sorted(d.glob(f'{r.video_id}__f*.jpg'))
    try: paths=[str(p) for p in done] if done else faces(r.video_path,d,r.video_id)
    except Exception as e: failed.append((r.video_id,'video',str(e))); paths=[]
    rows+=[dict(frame_path=p,video_id=r.video_id,group_id=r.group_id,
                label_visual=r.label_visual,label_audio=r.label_audio,
                type=r.type,partition=r.partition) for p in paths]
    try: a=mfcc(r.video_path)
    except Exception as e: failed.append((r.video_id,'audio',str(e))); a=np.zeros(N_MFCC*2,dtype='float32')
    aud.append(a)
    if n%100==0: print(f'{n}/{len(man)} | {len(rows)} frames | {(time.time()-t0)/60:.0f}m',flush=True)

idx=pd.DataFrame(rows); idx.to_csv(OUT/'frames'/'fakeav_frames.csv',index=False)
A=np.vstack(aud)
np.savez_compressed(AU/'fakeav_audio.npz',X=A,video_id=man.video_id.to_numpy(),
                    y_visual=man.label_visual.to_numpy(),y_audio=man.label_audio.to_numpy(),
                    g=man.group_id.to_numpy(),partition=man.partition.to_numpy())
print(f'\n{len(idx)} frames from {idx.video_id.nunique()} clips | audio {A.shape} | '
      f'{len(failed)} failures')
print(idx.groupby(['partition','label_visual']).size())
print('zero audio vectors:', int((A==0).all(1).sum()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1000 clips
100/1000 | 2000 frames | 3m
200/1000 | 4000 frames | 7m
300/1000 | 5990 frames | 10m
400/1000 | 7990 frames | 14m
500/1000 | 9990 frames | 18m
600/1000 | 11987 frames | 22m
700/1000 | 13987 frames | 26m
800/1000 | 15987 frames | 31m
900/1000 | 17986 frames | 36m
1000/1000 | 19986 frames | 41m

19986 frames from 1000 clips | audio (1000, 48) | 0 failures
partition  label_visual
test       0               1720
           1               1136
train      0               8350
           1               5720
val        0               1920
           1               1140
dtype: int64
zero audio vectors: 0


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time, random, json
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from sklearn.metrics import roc_auc_score

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic=True
OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026')
SIZE,BATCH,EPOCHS,PATIENCE,LR_HEAD,LR_BODY=300,32,5,3,1e-3,1e-4
BEST=OUT/'models'/'efficientnetv2s_fakeav_clean.pth'
CKPT=OUT/'models'/'fakeav_last.pth'

idx=pd.read_csv(OUT/'frames'/'fakeav_frames.csv')
tr,va=idx[idx.partition=='train'],idx[idx.partition=='val']
assert not set(tr.group_id)&set(va.group_id)
print(f'train {len(tr)} frames / {tr.group_id.nunique()} identities | val {len(va)}',flush=True)

TTF=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(.1,.1,.1),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
ETF=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class DS(Dataset):
    def __init__(s,df,tf): s.p=df.frame_path.tolist(); s.y=df.label_visual.tolist(); s.tf=tf
    def __len__(s): return len(s.p)
    def __getitem__(s,i):
        for _ in range(3):
            try: return s.tf(Image.open(s.p[i]).convert('RGB')), s.y[i]
            except OSError: time.sleep(1)
        return torch.zeros(3,SIZE,SIZE), s.y[i]

dev='cuda'
m=efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
m.classifier[1]=nn.Linear(m.classifier[1].in_features,2); m=m.to(dev)
head=list(m.classifier.parameters()); hid={id(p) for p in head}
opt=torch.optim.AdamW([{'params':[p for p in m.parameters() if id(p) not in hid],'lr':LR_BODY},
                       {'params':head,'lr':LR_HEAD}],weight_decay=1e-4)
crit=nn.CrossEntropyLoss(); sc=torch.amp.GradScaler('cuda')
start,best,bad,hist=1,-1,0,[]
if CKPT.exists():
    ck=torch.load(CKPT,map_location=dev,weights_only=False)
    m.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sc.load_state_dict(ck['scaler'])
    start,best,bad,hist=ck['epoch']+1,ck['best'],ck['bad'],ck['hist']
    print(f'resumed after epoch {ck["epoch"]}',flush=True)

g=torch.Generator(); g.manual_seed(SEED)
dtr=DataLoader(DS(tr,TTF),batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True,
               generator=g,drop_last=True,persistent_workers=True)
dva=DataLoader(DS(va,ETF),batch_size=BATCH*2,num_workers=2,pin_memory=True)

for ep in range(start,EPOCHS+1):
    m.train(); t0=time.time(); L=[]
    for k,(x,y) in enumerate(dtr,1):
        x,y=x.to(dev,non_blocking=True),y.to(dev,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'): loss=crit(m(x),y)
        sc.scale(loss).backward(); sc.step(opt); sc.update(); L.append(loss.item())
        if k%100==0: print(f'  ep{ep} {k}/{len(dtr)} loss {np.mean(L[-100:]):.4f} '
                           f'{(time.time()-t0)/60:.0f}m',flush=True)
    m.eval(); S,Y=[],[]
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for x,y in dva:
            S+=torch.softmax(m(x.to(dev)),1)[:,1].float().cpu().tolist(); Y+=y.tolist()
    auc=roc_auc_score(Y,S)
    hist.append(dict(epoch=ep,loss=float(np.mean(L)),val_auc=float(auc),
                     mins=round((time.time()-t0)/60,1)))
    print(hist[-1],flush=True)
    if auc>best:
        best,bad=auc,0
        torch.save({'model_state':m.state_dict(),'val_auc':float(auc),'epoch':ep},BEST)
        print('  new best saved',flush=True)
    else: bad+=1
    torch.save({'model':m.state_dict(),'opt':opt.state_dict(),'scaler':sc.state_dict(),
                'epoch':ep,'best':best,'bad':bad,'hist':hist},CKPT)
    json.dump({'best_val_auc':best,'history':hist,'train_frames':len(tr),
               'train_identities':int(tr.group_id.nunique()),'seed':SEED,
               'gpu':torch.cuda.get_device_name(0)},
              open(OUT/'results'/'fakeav_finetune.json','w'),indent=2)
    if bad>=PATIENCE: print('early stop'); break
print('best val AUC %.4f'%best)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train 14070 frames / 350 identities | val 3060
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 173MB/s]


  ep1 100/439 loss 0.1488 0m
  ep1 200/439 loss 0.0307 1m
  ep1 300/439 loss 0.0172 1m
  ep1 400/439 loss 0.0141 1m
{'epoch': 1, 'loss': 0.04874256080131053, 'val_auc': 0.9999735014619883, 'mins': 1.6}
  new best saved
  ep2 100/439 loss 0.0090 0m
  ep2 200/439 loss 0.0062 1m
  ep2 300/439 loss 0.0015 1m
  ep2 400/439 loss 0.0034 1m
{'epoch': 2, 'loss': 0.004961091159484381, 'val_auc': 0.9999990862573099, 'mins': 1.4}
  new best saved
  ep3 100/439 loss 0.0010 0m
  ep3 200/439 loss 0.0090 1m
  ep3 300/439 loss 0.0100 1m
  ep3 400/439 loss 0.0085 1m
{'epoch': 3, 'loss': 0.007921796495727091, 'val_auc': 0.999905884502924, 'mins': 1.4}
  ep4 100/439 loss 0.0056 0m
  ep4 200/439 loss 0.0052 1m
  ep4 300/439 loss 0.0055 1m
  ep4 400/439 loss 0.0029 1m
{'epoch': 4, 'loss': 0.004792031325009527, 'val_auc': 0.9998572277046783, 'mins': 1.4}
  ep5 100/439 loss 0.0005 0m
  ep5 200/439 loss 0.0042 1m
  ep5 300/439 loss 0.0065 1m
  ep5 400/439 loss 0.0025 1m
{'epoch': 5, 'loss': 0.00339708024114030

In [ ]:
import pandas as pd
idx = pd.read_csv(OUT/'frames'/'fakeav_frames.csv')
te = idx[idx.partition=='test']
print(te.groupby(['type','label_visual','label_audio']).agg(
    frames=('frame_path','size'), clips=('video_id','nunique')))

                                              frames  clips
type                label_visual label_audio               
FakeVideo-FakeAudio 1            1               557     28
FakeVideo-RealAudio 1            0               579     29
RealVideo-FakeAudio 0            1               220     11
RealVideo-RealAudio 0            0              1500     75


Audio features as described in Section 3.3.2.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, librosa, subprocess, tempfile, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026')
man=pd.read_csv(OUT/'manifests'/'fakeavceleb_manifest.csv'); SR=16000; WIN=4*SR

def feat48(video):
    try:
        with tempfile.NamedTemporaryFile(suffix='.wav') as t:
            subprocess.run(['ffmpeg','-y','-i',str(video),'-ac','1','-ar',str(SR),t.name],
                           capture_output=True)
            y,_=librosa.load(t.name,sr=SR)
        y=np.pad(y,(0,max(0,WIN-len(y))))[:WIN]
        m=librosa.feature.mfcc(y=y,sr=SR,n_mfcc=40).mean(1)
        sc=librosa.feature.spectral_contrast(y=y,sr=SR,n_bands=6).mean(1)
        z=np.array([librosa.feature.zero_crossing_rate(y).mean()])
        return np.concatenate([m,sc,z]).astype('float32')
    except Exception:
        return np.zeros(48,dtype='float32')

t0=time.time(); out=[]
with ProcessPoolExecutor(max_workers=2) as ex:
    for i,v in enumerate(ex.map(feat48,man.video_path.tolist(),chunksize=8),1):
        out.append(v)
        if i%200==0: print(f'  {i}/{len(man)} {(time.time()-t0)/60:.0f}m',flush=True)
A=np.vstack(out)
np.savez_compressed(OUT/'features'/'fakeav_audio_v2.npz',X=A,
                    video_id=man.video_id.to_numpy(),y_visual=man.label_visual.to_numpy(),
                    y_audio=man.label_audio.to_numpy(),g=man.group_id.to_numpy(),
                    partition=man.partition.to_numpy())
print(f'\n{A.shape} | zero vectors {int((A==0).all(1).sum())} | {(time.time()-t0)/60:.0f} min')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  200/1000 2m
  400/1000 4m
  600/1000 6m
  800/1000 8m
  1000/1000 10m

(1000, 48) | zero vectors 0 | 10 min


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, json, joblib, itertools
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026')
RES,MOD,FEA=OUT/'results',OUT/'models',OUT/'features'
z=np.load(FEA/'fakeav_visual.npz',allow_pickle=True); V={k:z[k] for k in z.files}
a=np.load(FEA/'fakeav_audio_v2.npz',allow_pickle=True)
order={v:i for i,v in enumerate(a['video_id'])}
A=a['X'][[order[v] for v in V['vid']]]
part,g=V['partition'],V['g']
tr,va,te=(part=='train'),(part=='val'),(part=='test')
assert not set(g[tr])&set(g[te])
print('visual',V['X'].shape,'audio',A.shape)

GRID=[dict(C=c,gamma=gm,class_weight=cw) for c,gm,cw in
      itertools.product([0.01,0.1,1,10,100],['scale','auto',1e-4,1e-3,1e-2],[None,'balanced'])]
def fit(X,y,name):
    sc=StandardScaler().fit(X[tr]); Xs=sc.transform(X)
    best,bauc=None,-1
    for p in GRID:
        try: auc=roc_auc_score(y[va],SVC(kernel='rbf',**p).fit(Xs[tr],y[tr]).decision_function(Xs[va]))
        except Exception: continue
        if auc>bauc: best,bauc=p,auc
    clf=SVC(kernel='rbf',**best).fit(Xs[tr],y[tr])
    print(f'{name}: {best} val AUC {bauc:.4f}',flush=True)
    joblib.dump({'scaler':sc,'svm':clf},MOD/f'fakeav_{name}_v2.pkl')
    return clf.decision_function(Xs), best, bauc

s_vis,p_vis,v_vis=fit(V['X'],V['y_visual'],'visual')
s_aud,p_aud,v_aud=fit(A,V['y_audio'],'audio')
s_cat,p_cat,v_cat=fit(np.hstack([V['X'],A]),V['y_visual'],'fusion_feature')

def M(y,s,tag):
    p=(s>0).astype(int); tn,fp,fn,tp=confusion_matrix(y,p).ravel()
    r=dict(condition=tag,auc=roc_auc_score(y,s),accuracy=accuracy_score(y,p),
           balanced_accuracy=(tp/(tp+fn)+tn/(tn+fp))/2,fake_detection_rate=tp/(tp+fn),
           false_positive_rate=fp/(fp+tn),confusion_matrix=[[int(tn),int(fp)],[int(fn),int(tp)]])
    print(f"{tag:34s} AUC {r['auc']:.4f} | acc {r['accuracy']:.4f} | "
          f"bal.acc {r['balanced_accuracy']:.4f}",flush=True)
    return r

yv,ya=V['y_visual'][te],V['y_audio'][te]
res=[M(yv,s_vis[te],'visual only (visual label)'),
     M(ya,s_aud[te],'audio only (AUDIO label)'),
     M(yv,s_cat[te],'feature-level fusion')]
zs=lambda x:(x-x[tr].mean())/(x[tr].std()+1e-9)
zv,za=zs(s_vis),zs(s_aud)
res.append(M(yv,(zv+za)[te]/2,'score-level fusion (mean)'))
for gate in (0.5,1.0,1.5):
    sg=np.where(np.abs(zv)<gate,(zv+za)/2,zv)
    r=M(yv,sg[te],f'confidence-gated fusion (gate={gate})')
    r['n_gated']=int((np.abs(zv[te])<gate).sum()); res.append(r)

print('\nper-category mean scores (test):')
rows=[]
for t in np.unique(V['type'][te]):
    k=te&(V['type']==t)
    rows.append(dict(category=t,n=int(k.sum()),visual=float(zv[k].mean()),
                     audio=float(za[k].mean()),mean_fusion=float(((zv+za)/2)[k].mean())))
    print(f"  {t:22s} n={rows[-1]['n']:3d} | visual {rows[-1]['visual']:+.2f} | "
          f"audio {rows[-1]['audio']:+.2f} | mean-fusion {rows[-1]['mean_fusion']:+.2f}")

json.dump(dict(results=res,per_category=rows,
               features='40 MFCC + 7 spectral contrast + 1 ZCR (48-dim)',
               params=dict(visual=str(p_vis),audio=str(p_aud),fusion=str(p_cat)),
               test_clips=int(te.sum())),
          open(RES/'fakeav_ablation.json','w'),indent=2,default=float)
np.savez_compressed(RES/'fakeav_test_scores.npz',y_visual=yv,y_audio=ya,visual=s_vis[te],
                    audio=s_aud[te],fusion=s_cat[te],type=V['type'][te])
print('saved')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
visual (1000, 1280) audio (1000, 48)
visual: {'C': 0.01, 'gamma': 'scale', 'class_weight': None} val AUC 1.0000
audio: {'C': 10, 'gamma': 0.01, 'class_weight': 'balanced'} val AUC 0.9943
fusion_feature: {'C': 0.01, 'gamma': 'scale', 'class_weight': None} val AUC 1.0000
visual only (visual label)         AUC 1.0000 | acc 0.9930 | bal.acc 0.9912
audio only (AUDIO label)           AUC 0.9884 | acc 0.9790 | bal.acc 0.9776
feature-level fusion               AUC 1.0000 | acc 0.9930 | bal.acc 0.9912
score-level fusion (mean)          AUC 0.9776 | acc 0.9021 | bal.acc 0.9038
confidence-gated fusion (gate=0.5) AUC 1.0000 | acc 1.0000 | bal.acc 1.0000
confidence-gated fusion (gate=1.0) AUC 0.9916 | acc 0.9161 | bal.acc 0.9214
confidence-gated fusion (gate=1.5) AUC 0.9776 | acc 0.9021 | bal.acc 0.9038

per-category mean scores (test):
  FakeVideo-FakeAudio    n= 28 | vi

In [ ]:
import numpy as np, json
from pathlib import Path
from scipy import stats as sstats
RES=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026/results')
d=np.load(RES/'fakeav_test_scores.npz',allow_pickle=True); y=d['y_visual']
def midrank(x):
    o=np.argsort(x); sx=x[o]; n=len(x); r=np.zeros(n); i=0
    while i<n:
        j=i
        while j<n-1 and sx[j+1]==sx[i]: j+=1
        r[i:j+1]=0.5*(i+j)+1; i=j+1
    out=np.empty(n); out[o]=r; return out
def delong(y,a,b):
    pos=np.array([a[y==1],b[y==1]]); neg=np.array([a[y==0],b[y==0]])
    m,n=pos.shape[1],neg.shape[1]
    tx=np.array([midrank(pos[i]) for i in range(2)])
    ty=np.array([midrank(neg[i]) for i in range(2)])
    tz=np.array([midrank(np.concatenate([pos[i],neg[i]])) for i in range(2)])
    auc=tz[:,:m].sum(1)/m/n-(m+1)/(2*n)
    v01=(tz[:,:m]-tx)/n; v10=1-(tz[:,m:]-ty)/m
    S=np.cov(v01)/m+np.cov(v10)/n; L=np.array([[1,-1]])
    var=float(np.squeeze(L@S@L.T))
    if var<=0: return float(auc[0]),float(auc[1]),1.0
    z=float((auc[0]-auc[1])/np.sqrt(var))
    return float(auc[0]),float(auc[1]),float(2*(1-sstats.norm.cdf(abs(z))))
zs=lambda x:(x-x.mean())/(x.std()+1e-9)
out={}
for tag,other in [('feature_fusion',d['fusion']),('score_mean',(zs(d['visual'])+zs(d['audio']))/2)]:
    A,B,p=delong(y,d['visual'],other)
    out[f'visual_vs_{tag}']=dict(auc_visual=A,auc_other=B,p_value=p)
    print(f'DeLong visual ({A:.4f}) vs {tag} ({B:.4f}): p = {p:.4f}')
json.dump(out,open(RES/'fakeav_delong.json','w'),indent=2)
print('saved')

DeLong visual (1.0000) vs feature_fusion (1.0000): p = 1.0000
DeLong visual (1.0000) vs score_mean (0.9798): p = 0.0159
saved


In [ ]:
import joblib
from pathlib import Path
MOD=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026/models')
for n in ['fakeav_visual_v2','fakeav_audio_v2','fakeav_fusion_feature_v2']:
    m=joblib.load(MOD/f'{n}.pkl')['svm']
    print(n, m.n_support_.tolist(), int(m.n_support_.sum()))

fakeav_visual_v2 [211, 209] 420
fakeav_audio_v2 [105, 85] 190
fakeav_fusion_feature_v2 [221, 218] 439
